# Sequential training. 

This scripts takes the  the original annotation (for step 1) and the corrected additional labels and trains the models again. 

In [1]:
sequence = 1
from opensoundscape import Audio, Spectrogram
from opensoundscape.annotations import BoxedAnnotations

import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt

# for step 1 only

In [2]:
# load the annotation file from Raven
annotation_file = Path(r"/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/annotations_airguns150.txt")
audio_file = Path(r"/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Catalog AMARs/Airgun/all_together.wav") 


In [3]:
annot_list = [annotation_file]
audio_list = [audio_file]

# Create an object from Raven file
annotations_og = BoxedAnnotations.from_raven_files(
    raven_files=annot_list,
    audio_files=audio_list,
    annotation_column=None,  # Put None if you don't have an annotation column
)

# Inspect the object's .df attribute
# which contains the table of annotations
annotations_og.df.head()
annotations_og.df["annotation"] = "airgun"


# Parameters to use for label creation
clip_duration = 5
clip_overlap = 0
min_label_overlap = 0.25

# Create dataframe of one-hot labels
clip_labels = annotations_og.clip_labels(
    clip_duration = clip_duration,
    clip_overlap = clip_overlap,
    min_label_overlap = min_label_overlap,
)

Load the annotations from the inference step

,audio_file,annotation_file,annotation,start_time,end_time,low_f,high_f
0,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,0.0,5.0,0,1000
1,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,5.0,10.0,0,1000
2,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,0.0,5.0,0,1000
3,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,0.0,5.0,0,1000
4,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,5.0,10.0,0,1000
...,...,...,...,...,...,...,...
68,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,5.0,10.0,0,1000
69,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,5.0,10.0,0,1000
70,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,5.0,10.0,0,1000
71,/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Ca...,NaN,airgun,0.0,5.0,0,1000


In [11]:
seq_annotations_folder=f"annotations_step{sequence}"
annotations_new = BoxedAnnotations.from_csv(Path(seq_annotations_folder).joinpath("ML_annotations_step.csv"))
annotations_new.df['annotation']='airgun'
clip_labels_new = annotations_new.clip_labels(
    clip_duration = clip_duration,
    clip_overlap = clip_overlap,
    min_label_overlap = min_label_overlap,
    audio_files=annotations_new.df['audio_file'].unique()
)


Join the previous step and the current step dataframe

In [14]:
clip_labels_full = pd.concat([clip_labels, clip_labels_new])

In [16]:
labels = clip_labels_full.sort_index(level="start_time")
n = len(labels)
print(n)
train_df = labels.iloc[: int(0.6 * n)]
val_df   = labels.iloc[int(0.6 * n) :]
# test_df  = labels.iloc[int(0.85 * n) :]

for name, d in [("train", train_df), ("val", val_df),]:
    print(name, len(d), "clips,",d["airgun"].sum(), "with airgun")

429
train 257 clips, 140 with airgun
val 172 clips, 134 with airgun


Load the models and train the resnet and the efficien net

In [19]:
# Create a CNN object designed to recognize 3-second samples
from opensoundscape import CNN

# Use resnet34 architecture
resnet = "resnet18"
effnet = "efficientnet_b0"

# Can use this code to get your classes, if needed
class_list = list(train_df.columns)
class_list
model1 = CNN(
    architecture=resnet,
    classes=class_list,
    sample_duration=clip_duration,  # 10s, selected above
    sample_rate=6000,
)

model2 = CNN(
    architecture=effnet,
    classes=class_list,
    sample_duration=clip_duration,  # 10s, selected above
    sample_rate=6000,
)


print(model1.device, model2.device)

cuda:0 cuda:0


Configure the wandb session

In [20]:
import wandb
from Keys import api_key

# try:
wandb.login()
wandb_session = wandb.init(
    entity="AI_for_Ecology_ITAW",  # replace with your entity/group name
    project="Airgun_detector",  # replace with your project name
    name="First iteration",
)
# except:  # if wandb.init fails, don't use wandb logging
#     print("failed to create wandb session. wandb session will be None")
#     wandb_session = None

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/biofelip/.netrc.
wandb: Currently logged in as: felipe-1092 (AI_for_Ecology_ITAW) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [25]:
checkpoint_folder = Path(seq_annotations_folder)

model1.train(train_df,
    val_df,
    steps=300,
    batch_size=16,
    log_interval=2,  # log progress every 100 batches
    validation_interval=2,
    # num_workers=num_workers,  # parallelized cpu tasks for preprocessing
    wandb_session=wandb_session,
    save_path=checkpoint_folder)  # location to save checkpoints)

wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.


  0%|          | 0/300 [00:00<?, ?it/s]

Step: 0/300 (0.0%)
	Running Average Loss: 0.763
	Most Recent Step Loss: 0.763

Validation.


  0%|          | 0/11 [00:00<?, ?it/s]

Step: 2/300 (0.7%)
	Running Average Loss: 0.724
	Most Recent Step Loss: 0.638

Validation.


...

Step: 296/300 (98.7%)
	Running Average Loss: 0.463
	Most Recent Step Loss: 0.500

Validation.


  0%|          | 0/11 [00:00<?, ?it/s]

Step: 298/300 (99.3%)
	Running Average Loss: 0.507
	Most Recent Step Loss: 0.331

Validation.


  0%|          | 0/11 [00:00<?, ?it/s]


Best Model Appears at Step 154 with Validation score 0.929.


PosixPath('annotations_step1/effnet')

In [ ]:
effnet_folder = Path(checkpoint_folder).joinpath("effnet")
effnet_folder.mkdir(exist_ok=True)
model2.train(train_df,
    val_df,
    steps=300,
    batch_size=16,
    log_interval=2,  # log progress every 100 batches
    validation_interval=2,
    # num_workers=num_workers,  # parallelized cpu tasks for preprocessing
    wandb_session=wandb_session,
    save_path=effnet_folder)  # location to save checkpoints)

In [29]:
import bioacoustics_model_zoo as bmz
perch = bmz.Perch2ONNX(headless=True)
train_emb=np.asanyarray(perch.embed(train_df, return_dfs=False, return_preds=False))
val_emb=perch.embed(val_df, return_dfs=False, return_preds=False)

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:2930: UserWarning: The columns of input samples df differ from `model.classes`. Discarding sample df columns.
  warnings.warn(


  0%|          | 0/257 [00:00<?, ?it/s]

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:2930: UserWarning: The columns of input samples df differ from `model.classes`. Discarding sample df columns.
  warnings.warn(


  0%|          | 0/172 [00:00<?, ?it/s]

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
# 3. Small classifier on top
clf = LogisticRegression(max_iter=5000, class_weight="balanced")   # balanced: you have few negatives
clf.fit(train_emb, train_df['airgun'])

val_scores = clf.predict_proba(val_emb)[:, 1]
print("val AUC:", roc_auc_score(val_df["airgun"], val_scores))


val AUC: 0.856637863315004


save the shallwo classifier

In [ ]:
import joblib
joblib.dump(clf, Path(checkpoint_folder).joinpath("Shallow_classifier.joblib"))   # save

['annotations_step1/effnet/Shallow_classifier.joblib']